# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset, "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells", using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Cite as: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview

Let's review the available record sets, fields, and their `@id` values.

Entities such as record sets, fields, and columns are always referenced by their `@id` for consistency and reproducibility.

In [ ]:
# List all record sets in the dataset

print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}\n  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (Field @id: {field.id}, DataType: {field.data_type})")
    print()

> **Tip:** Use the `@id` values above to access specific record sets or fields for your analysis.

In [ ]:
# Preview records from each record set

for rs in metadata.record_sets:
    print(f"First 1-2 records from record set '{rs.name}' (@id: {rs.id}):")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        print(json.dumps(record, indent=2))
        if i >= 1:
            break
    print()

## 3. Data Extraction

Let's extract data from one or more record sets using their `@id` values, and load them into pandas DataFrames for analysis.

In [ ]:
# Collect all record set @ids
record_sets_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

# Load each record set into a DataFrame
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display DataFrame columns for the first record set
if record_sets_ids:
    first_rs = record_sets_ids[0]
    print(f"Columns in record set '{first_rs}':")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate some basic EDA using one of the numeric fields from the DataFrame. We'll filter, normalize, and group the data. Make sure to substitute the appropriate `@id` strings and field names as needed.

In [ ]:
# --- Customize these IDs for your data exploration ---
# For this example, we use the first record set and one of its numeric fields

selected_record_set_id = record_sets_ids[0]  # Or choose a specific one
df = dataframes[selected_record_set_id]

# Choose a numeric field's @id (and the actual column, which may be the field name)
# Here we look for a field with a numeric type
numeric_field_id = None
numeric_field_name = None
group_field_id = None
for field in metadata.record_sets[0].fields:
    if field.data_type in ('Float', 'Integer', 'Number') and not numeric_field_id:
        numeric_field_id = field.id
        numeric_field_name = field.name
    if field.data_type == 'Text' and not group_field_id:
        group_field_id = field.id
        group_field_name = field.name

print(f"Selected numeric field: {numeric_field_name} (@id: {numeric_field_id})")
print(f"Selected group field: {group_field_name} (@id: {group_field_id})")

# Proceed only if data exists
if numeric_field_name in df.columns:
    # Convert to numeric (handling non-numeric, if necessary)
    df[numeric_field_name] = pd.to_numeric(df[numeric_field_name], errors='coerce')
    threshold = df[numeric_field_name].mean()
    filtered_df = df[df[numeric_field_name] > threshold]

    print(f"Filtered records with {numeric_field_name} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_name}_normalized"] = (
        (filtered_df[numeric_field_name] - filtered_df[numeric_field_name].mean()) /
        filtered_df[numeric_field_name].std()
    )

    print(f"Normalized {numeric_field_name} for filtered records:")
    display(filtered_df[[numeric_field_name, f"{numeric_field_name}_normalized"]].head())

    # Group by a text field (if appropriate)
    if group_field_name and group_field_name in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_name)[numeric_field_name].mean().reset_index()
        print(f"Grouped data by {group_field_name} (mean {numeric_field_name}):")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA in this record set.")

## 5. Visualization

Now, let's visualize the distribution of the selected numeric field and its relationship with the group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if the numeric field is appropriate for plotting
if numeric_field_name and numeric_field_name in df.columns and df[numeric_field_name].dtype in ['float64', 'int64']:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_name].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_name}")
    plt.xlabel(numeric_field_name)
    plt.ylabel("Count")
    plt.show()

    if group_field_name and group_field_name in df.columns:
        plt.figure(figsize=(10, 4))
        top_groups = df[group_field_name].value_counts().head(10).index
        plot_df = df[df[group_field_name].isin(top_groups)]
        sns.boxplot(data=plot_df, x=group_field_name, y=numeric_field_name)
        plt.title(f"{numeric_field_name} by {group_field_name} (Top 10 Groups)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough numeric data available for plotting.")

## 6. Conclusion

In this notebook, we've demonstrated how to load and explore a FAIR² dataset described by a Croissant schema using the `mlcroissant` library. We've:

- Inspected available record sets and their fields by `@id`
- Loaded the records for detailed analysis into pandas DataFrames
- Applied exploratory data operations, including filtering and normalization
- Visualized numeric field distributions and group comparisons

You can further customize this notebook to perform your own analyses or prepare data for machine learning workflows.